In [ ]:
pip install -q tiktoken 

In [1]:
import tiktoken
import torch
import numpy as np

c:\Users\Vivek\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


In [2]:
tokenizer=tiktoken.get_encoding("cl100k_base")

In [3]:
tokenizer.encode("hello world")

[15339, 1917]

In [4]:
text="""
We’ve seen that Unicode code points are abstractly identified by their index in the codespace, ranging from U+0000 to U+10FFFF. But how do code points get represented as bytes, in memory or in a file?

The most convenient, computer-friendliest (and programmer-friendliest) thing to do would be to just store the code point index as a 32-bit integer. This works, but it consumes 4 bytes per code point, which is sort of a lot. Using 32-bit ints for Unicode will cost you a bunch of extra storage, memory, and performance in bandwidth-bound scenarios, if you work with a lot of text.

Consequently, there are several more-compact encodings for Unicode. The 32-bit integer encoding is officially called UTF-32 (UTF = “Unicode Transformation Format”), but it’s rarely used for storage. At most, it comes up sometimes as a temporary internal representation, for examining or operating on the code points in a string.

Much more commonly, you’ll see Unicode text encoded as either UTF-8 or UTF-16. These are both variable-length encodings, made up of 8-bit or 16-bit units, respectively. In these schemes, code points with smaller index values take up fewer bytes, which saves a lot of memory for typical texts. The trade-off is that processing UTF-8/16 texts is more programmatically involved, and likely slower.
"""

In [5]:
input_sequences=[]
for line in text.split('\n'):
  token_list=tokenizer.encode(line)
  for i in range(1,len(token_list)):
    n_gram_seq=token_list[:i+1]
    input_sequences.append(n_gram_seq)

In [6]:
input_sequences[:10]

[[1687, 4070],
 [1687, 4070, 3970],
 [1687, 4070, 3970, 430],
 [1687, 4070, 3970, 430, 36997],
 [1687, 4070, 3970, 430, 36997, 2082],
 [1687, 4070, 3970, 430, 36997, 2082, 3585],
 [1687, 4070, 3970, 430, 36997, 2082, 3585, 527],
 [1687, 4070, 3970, 430, 36997, 2082, 3585, 527, 8278],
 [1687, 4070, 3970, 430, 36997, 2082, 3585, 527, 8278, 398],
 [1687, 4070, 3970, 430, 36997, 2082, 3585, 527, 8278, 398, 11054]]

In [7]:
max_seq_len=max([len(x) for x in input_sequences])
max_seq_len

89

In [8]:
len(input_sequences)

288

In [9]:
tokenizer.decode(input_sequences[0])

'We’ve'

In [10]:
pad_seq=[]
for seq in input_sequences:
  pad_seq.append(np.pad(seq,(max_seq_len-len(seq),0)))
pad_seq=np.array(pad_seq)

In [11]:
pad_seq[:5]

array([[    0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,  1687,  4070],
       [    0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            

In [12]:
max([len(x) for x in pad_seq])

89

In [13]:
pad_seq[-1].shape

(89,)

In [14]:
x=pad_seq[:,:-1]
x.shape

(288, 88)

In [15]:
y=pad_seq[:,-1]
y.shape


(288,)

In [16]:
tokenizer.n_vocab

100277

In [17]:
from torch.utils.data import TensorDataset, DataLoader

In [18]:
inputs=torch.tensor(x)

In [19]:
targets=torch.LongTensor(y)

In [20]:
dataset=TensorDataset(inputs,targets)

In [21]:
dataloader=DataLoader(dataset,batch_size=32,shuffle=True,num_workers=15,persistent_workers=True)

In [22]:
for i,j in dataloader:
  print(i.shape)
  print(j.shape)
  break

torch.Size([32, 88])
torch.Size([32])


In [ ]:
pip install -q lightning

In [23]:
import torch.nn as nn
import lightning as L
from torch.optim import Adam

c:\Users\Vivek\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [24]:
device="cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [27]:
class Model(nn.Module): # Use L.LightninigModule for Using Lightning
  def __init__(self) -> None:
    super().__init__()
    self.embd=nn.Embedding(100277,100)
    self.lstm=nn.LSTM(100,100,3,dropout=0.2,batch_first=True)
    self.ll1=nn.Linear(100,10)
    self.ll2=nn.Linear(10,100277)
    self.loss_fn=nn.CrossEntropyLoss()
    self.optimizer=Adam(self.parameters())
    self.loss=[]

  def forward(self,x):
    embd=self.embd(x)
    out,_=self.lstm(embd)
    logits=self.ll1(out[:,-1,:])
    logits=self.ll2(logits)
    return logits

  def configure_optimizers(self):
    return Adam(self.parameters())

  def fitting(self,dataloader):
    self.train()
    for batch,(x,y) in enumerate(dataloader):
      x,y=x.to(device),y.to(device)

      pred=self.forward(x)
      loss=self.loss_fn(pred,y)
      self.loss.append(loss)
      loss.backward()
      self.optimizer.step()
      self.optimizer.zero_grad()

  def training_step(self, batch, batch_idx):
    input_i, label_i=batch
    output_i=self.forward(input_i)
    loss=self.loss_fn(output_i, label_i)
    self.loss.append(loss)
    return loss

  def predict(self,tokens):
    with torch.no_grad():
      return self.forward(tokens)



In [28]:
model=Model()
model

Model(
  (embd): Embedding(100277, 100)
  (lstm): LSTM(100, 100, num_layers=3, batch_first=True, dropout=0.2)
  (ll1): Linear(in_features=100, out_features=10, bias=True)
  (ll2): Linear(in_features=10, out_features=100277, bias=True)
  (loss_fn): CrossEntropyLoss()
)

In [29]:
model.to(device)

Model(
  (embd): Embedding(100277, 100)
  (lstm): LSTM(100, 100, num_layers=3, batch_first=True, dropout=0.2)
  (ll1): Linear(in_features=100, out_features=10, bias=True)
  (ll2): Linear(in_features=10, out_features=100277, bias=True)
  (loss_fn): CrossEntropyLoss()
)

In [82]:
#Simple Training Using Pytorch
for epoch in range(10):
    model.fitting(dataloader)
    print(f"------------------------------------------------------------------------------------")
    print(f"epoch:{epoch+1}")
    print(f"loss:{model.loss[-1]}")

------------------------------------------------------------------------------------
epoch:1
loss:9.436872482299805
------------------------------------------------------------------------------------
epoch:2
loss:6.9268412590026855
------------------------------------------------------------------------------------
epoch:3
loss:7.0810227394104
------------------------------------------------------------------------------------
epoch:4
loss:7.4951958656311035
------------------------------------------------------------------------------------
epoch:5
loss:6.481817722320557
------------------------------------------------------------------------------------
epoch:6
loss:5.139916896820068
------------------------------------------------------------------------------------
epoch:7
loss:5.292392730712891
------------------------------------------------------------------------------------
epoch:8
loss:6.274486541748047
------------------------------------------------------------------------

In [ ]:
#Training Using Lightning
trainer=L.Trainer(max_epochs=100,accelerator='auto',devices='auto')

In [ ]:
trainer.fit(model,train_dataloaders=dataloader)

In [ ]:
model.loss[-1]

In [ ]:
checkpoint=trainer.checkpoint_callback.best_model_path # Saving Checkpoint For Continuing

In [ ]:
trainer=L.Trainer(max_epochs=200,accelerator='auto',devices='auto')

In [ ]:
trainer.fit(model,train_dataloaders=dataloader,ckpt_path=checkpoint)

In [32]:
def predict(text):
  with torch.no_grad():
    tokens=tokenizer.encode(text)
    logits=model.forward(torch.tensor([tokens]).to(device))
    prob=logits.softmax(dim=1)
    return tokenizer.decode([(prob.argmax().item())])

In [37]:
model.eval()
predict("The most convenient, computer-friendliest (and programmer-friendliest) thing to do would be to just store the code point index as a 32-bit integer. This works, but it consumes 4 bytes per code point, which is sort of a lot. Using 32-bit ints for Unicode will cost you a bunch of extra storage, memory, and performance in bandwidth-bound scenarios, if you work with a lot")

' of'

In [ ]:
# Saving Model and Weights
torch.save(model, "model.pth")
torch.save(model.state_dict(), "model_weights.pth")

In [30]:
#Loading Model Weights
model.load_state_dict(torch.load('model_weights.pth', weights_only=True))

<All keys matched successfully>